# STT Model Comparison: Statistical Significance

Compares a candidate STT run against a baseline using paired statistical tests.

**Requires N ≥ 5 samples per metric for Wilcoxon signed-rank test.** With fewer samples, the notebook reports descriptive statistics only and flags that inference is not valid.

Set environment variables before running (or edit the cells below):
- `CANDIDATE_RUN_ID` — UUID of the candidate (quality) run
- `BASELINE_RUN_ID` — UUID of the baseline (turbo) run

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)

CANDIDATE_RUN_ID = os.environ.get("CANDIDATE_RUN_ID", "9835ed61-9906-449d-bb8b-365cf7b0e70c")
BASELINE_RUN_ID  = os.environ.get("BASELINE_RUN_ID",  "87bac619-f810-4994-8a27-7b883b8f99b7")
ALPHA = 0.05  # significance threshold

# Metrics where lower value is better (error metrics)
LOWER_IS_BETTER = {"wer", "cer", "mer", "wil", "cp_wer"}
# Metrics where higher value is better
HIGHER_IS_BETTER = {"bertscore_f1", "bertscore_ref_f1"}

print(f"Candidate : {CANDIDATE_RUN_ID}")
print(f"Baseline  : {BASELINE_RUN_ID}")
print(f"Alpha     : {ALPHA}")

In [ ]:
from analysis.eval_insights import fetch_eval_rows, fetch_run_meta

meta = fetch_run_meta(CANDIDATE_RUN_ID, BASELINE_RUN_ID)
if not meta.empty:
    display(meta[["run_id", "model_name", "run_scope", "run_timestamp"]].set_index("run_id"))

raw = fetch_eval_rows(CANDIDATE_RUN_ID, BASELINE_RUN_ID)
print(f"\nLoaded {len(raw)} evaluation rows")

In [ ]:
# ---------------------------------------------------------------------------
# Build paired dataset: one row per (sample_id, metric_name)
# with candidate_value and baseline_value side by side.
# ---------------------------------------------------------------------------

# Normalise run_id column from details JSONB
if "details.run_id" in raw.columns:
    raw["_run_id"] = raw["details.run_id"].fillna(raw.get("details_run_id", "")).fillna("")
elif "details_run_id" in raw.columns:
    raw["_run_id"] = raw["details_run_id"].fillna("")
else:
    raw["_run_id"] = ""

# Keep only rows explicitly tagged as candidate or baseline
candidate_rows = raw[raw["_run_id"] == CANDIDATE_RUN_ID][["sample_id", "metric_name", "metric_value"]].copy()
candidate_rows = candidate_rows.rename(columns={"metric_value": "candidate"})

# Baseline may come from separate rows OR from ref_* columns in candidate rows
baseline_rows = raw[raw["_run_id"] == BASELINE_RUN_ID][["sample_id", "metric_name", "metric_value"]].copy()
baseline_rows = baseline_rows.rename(columns={"metric_value": "baseline"})

# If no explicit baseline rows, derive from ref_* columns
REF_COL_MAP = {"wer": "details.ref_wer", "cer": "details.ref_cer", "mer": "details.ref_mer", "wil": "details.ref_wil"}
if baseline_rows.empty:
    synthesised = []
    for metric, col in REF_COL_MAP.items():
        if col in candidate_rows.columns:
            sub = candidate_rows.loc[candidate_rows["metric_name"] == metric, ["sample_id"]].copy()
            sub["metric_name"] = metric
            sub["baseline"] = pd.to_numeric(candidate_rows.loc[candidate_rows["metric_name"] == metric, col], errors="coerce").values
            synthesised.append(sub)
    if synthesised:
        baseline_rows = pd.concat(synthesised, ignore_index=True)

paired = pd.merge(candidate_rows, baseline_rows, on=["sample_id", "metric_name"], how="inner")
paired["delta"] = paired["candidate"] - paired["baseline"]

metrics_available = sorted(paired["metric_name"].unique())
n_samples = paired["sample_id"].nunique()
print(f"Samples with paired data : {n_samples}")
print(f"Metrics available        : {metrics_available}")

if n_samples < 5:
    print(
        f"\n⚠  WARNING: N={n_samples} is below the minimum (5) for the Wilcoxon signed-rank test."
        "\n   Descriptive statistics are shown below, but statistical inference is NOT valid at this sample size."
        "\n   Re-run the pipeline with --limit 10 or more samples to enable significance testing."
    )

In [ ]:
# ---------------------------------------------------------------------------
# Descriptive statistics per metric
# ---------------------------------------------------------------------------

rows = []
for metric in metrics_available:
    sub = paired[paired["metric_name"] == metric].dropna(subset=["candidate", "baseline"])
    if sub.empty:
        continue
    n = len(sub)
    c_mean = sub["candidate"].mean()
    b_mean = sub["baseline"].mean()
    delta_mean = sub["delta"].mean()
    delta_std  = sub["delta"].std() if n > 1 else float("nan")
    delta_sem  = delta_std / np.sqrt(n) if n > 1 else float("nan")
    ci95_lo = delta_mean - 1.96 * delta_sem if n > 1 else float("nan")
    ci95_hi = delta_mean + 1.96 * delta_sem if n > 1 else float("nan")

    if metric in LOWER_IS_BETTER:
        winner = "candidate" if delta_mean < 0 else ("baseline" if delta_mean > 0 else "tie")
    elif metric in HIGHER_IS_BETTER:
        winner = "candidate" if delta_mean > 0 else ("baseline" if delta_mean < 0 else "tie")
    else:
        winner = "?"

    rows.append({
        "metric": metric,
        "n": n,
        "candidate_mean": round(c_mean, 4),
        "baseline_mean": round(b_mean, 4),
        "mean_delta": round(delta_mean, 4),
        "95%_ci": f"[{ci95_lo:.4f}, {ci95_hi:.4f}]" if not np.isnan(ci95_lo) else "N/A (N=1)",
        "winner": winner,
    })

desc_df = pd.DataFrame(rows).set_index("metric")
print("Descriptive Statistics (candidate − baseline)")
display(desc_df)

In [ ]:
# ---------------------------------------------------------------------------
# Wilcoxon signed-rank test (paired, non-parametric)
# Requires N >= 5. Falls back to t-test note at smaller N.
# ---------------------------------------------------------------------------
MIN_N_WILCOXON = 5

sig_rows = []
for metric in metrics_available:
    sub = paired[paired["metric_name"] == metric].dropna(subset=["candidate", "baseline"])
    if sub.empty:
        continue
    n = len(sub)
    c_vals = sub["candidate"].values
    b_vals = sub["baseline"].values
    diffs = c_vals - b_vals

    if n < MIN_N_WILCOXON:
        sig_rows.append({"metric": metric, "n": n, "test": "—", "statistic": "—", "p_value": "—",
                          "significant": f"N/A (need N≥{MIN_N_WILCOXON})", "note": "increase sample size"})
        continue

    if np.all(diffs == 0):
        sig_rows.append({"metric": metric, "n": n, "test": "wilcoxon", "statistic": 0.0,
                          "p_value": 1.0, "significant": False, "note": "all differences zero"})
        continue

    stat, p = stats.wilcoxon(c_vals, b_vals, alternative="two-sided")
    sig_rows.append({
        "metric": metric,
        "n": n,
        "test": "wilcoxon",
        "statistic": round(float(stat), 4),
        "p_value": round(float(p), 4),
        "significant": p < ALPHA,
        "note": f"p {'<' if p < ALPHA else '≥'} {ALPHA}",
    })

sig_df = pd.DataFrame(sig_rows).set_index("metric")
print(f"\nWilcoxon Signed-Rank Tests (α = {ALPHA}, two-sided)")
display(sig_df)

In [ ]:
# ---------------------------------------------------------------------------
# Bootstrap confidence intervals on the mean delta (1000 resamples)
# ---------------------------------------------------------------------------
N_BOOTSTRAP = 1000
rng = np.random.default_rng(seed=42)

boot_rows = []
for metric in metrics_available:
    sub = paired[paired["metric_name"] == metric].dropna(subset=["candidate", "baseline"])
    if len(sub) < 2:
        boot_rows.append({"metric": metric, "bootstrap_ci_95": "N/A (N=1)",
                           "boot_mean_delta": "—", "ci_excludes_zero": "—"})
        continue
    diffs = (sub["candidate"] - sub["baseline"]).values
    boot_means = [rng.choice(diffs, size=len(diffs), replace=True).mean() for _ in range(N_BOOTSTRAP)]
    lo, hi = np.percentile(boot_means, [2.5, 97.5])
    excludes_zero = not (lo <= 0 <= hi)
    boot_rows.append({
        "metric": metric,
        "boot_mean_delta": round(float(np.mean(boot_means)), 4),
        "bootstrap_ci_95": f"[{lo:.4f}, {hi:.4f}]",
        "ci_excludes_zero": excludes_zero,
    })

boot_df = pd.DataFrame(boot_rows).set_index("metric")
print(f"Bootstrap 95% CI on mean delta (candidate − baseline), {N_BOOTSTRAP} resamples")
print("ci_excludes_zero=True suggests the delta is unlikely to be noise (non-overlapping with zero).")
display(boot_df)

In [ ]:
# ---------------------------------------------------------------------------
# Per-metric delta distribution plots
# ---------------------------------------------------------------------------
if n_samples >= 2:
    n_metrics = len(metrics_available)
    ncols = min(3, n_metrics)
    nrows = (n_metrics + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), squeeze=False)
    axes_flat = axes.flatten()

    for idx, metric in enumerate(metrics_available):
        ax = axes_flat[idx]
        sub = paired[paired["metric_name"] == metric].dropna(subset=["delta"])
        if sub.empty:
            ax.set_visible(False)
            continue
        diffs = sub["delta"].values
        ax.hist(diffs, bins=max(5, len(diffs) // 3), color="steelblue", edgecolor="white", alpha=0.85)
        ax.axvline(0, color="black", linewidth=1.2, linestyle="--", label="no difference")
        ax.axvline(diffs.mean(), color="tomato", linewidth=1.5, label=f"mean={diffs.mean():.3f}")
        direction = "← candidate better" if metric in LOWER_IS_BETTER else "→ candidate better"
        ax.set_title(f"{metric}\n({direction})", fontsize=10)
        ax.set_xlabel("delta (candidate − baseline)")
        ax.set_ylabel("count")
        ax.legend(fontsize=8)

    for idx in range(n_metrics, len(axes_flat)):
        axes_flat[idx].set_visible(False)

    plt.suptitle("Delta Distribution per Metric (candidate − baseline)", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("Delta distribution plot requires N ≥ 2 samples. Run with --limit 10 or more.")

In [ ]:
# ---------------------------------------------------------------------------
# Summary interpretation
# ---------------------------------------------------------------------------
print("=" * 60)
print("INTERPRETATION GUIDE")
print("=" * 60)
print()
print("Wilcoxon signed-rank test: non-parametric paired test.")
print("  H0: the median delta between candidate and baseline = 0")
print(f"  Reject H0 if p < {ALPHA}. Rejection → the difference is unlikely due to chance.")
print()
print("Bootstrap CI: if the 95% interval excludes 0, the observed direction")
print("  of the delta is consistent across resamples (more reliable than a point estimate).")
print()
print("Practical significance vs statistical significance:")
print("  A statistically significant WER delta of 0.001 (0.1%) may not matter in production.")
print("  A non-significant delta of 0.05 (5%) on N=5 may matter but needs more data to confirm.")
print()
if n_samples < MIN_N_WILCOXON:
    print(f"⚠  Current N={n_samples}. Statistical tests require N ≥ {MIN_N_WILCOXON}.")
    print("   Increase --limit to 10+ in run_full_pipeline.py to enable full analysis.")